# EDA и бинарная классификация на датасете «Титаник»

**Дисциплина:** Машинное обучение и искусственный интеллект  
**Модуль 1:** Введение в Data Science и EDA  
**Датасет:** Titanic — Machine Learning from Disaster  

**Выполнил:** Станислав Федорушкин  
**Группа:** ПКТб-23-1  
**Преподаватель:** `[УКАЖИТЕ ФИО ПРЕПОДАВАТЕЛЯ]`  

**Ссылка на репозиторий:** https://github.com/Ogn1k/ml-course-fedorushkin  
**Путь к модулю:** `module-1-titanic/notebook.ipynb`  
**Дата выполнения:** 22.09.2026

## Введение

**Задача:** по сведениям о пассажире предсказать один из двух исходов: `0` — пассажир не выжил, `1` — выжил.

**Своими словами:** ML-проект — это не только обучение алгоритма, а последовательная проверка всей цепочки от качества исходных данных до полезности и воспроизводимости результата. Сначала данные исследуют и очищают, затем формируют признаки, обучают модели, проверяют их на отложенной выборке и сохраняют вместе со всеми преобразованиями.

**Почему это классификация:** целевая переменная принимает два дискретных значения, а не произвольное число на непрерывной шкале. Регрессия подошла бы, например, для прогноза стоимости билета, но не для выбора между классами «выжил» и «не выжил».

**Основная метрика:** ROC-AUC удобна для сравнения моделей, поскольку оценивает качество ранжирования пассажиров по вероятности выживания при всех порогах. При этом из-за умеренного дисбаланса классов отдельно рассматриваются F1, precision и recall; одной accuracy недостаточно, потому что тривиальный прогноз большинства уже даст около 61,6%.

**Цели работы:**

1. Провести полный EDA датасета «Титаник».
2. Обучить не менее двух моделей и стремиться к ROC-AUC не ниже 0,80.
3. Реализовать функцию предсказания для нового пассажира.

## Теоретическая часть

> **Важно: весь текст этого раздела сформулирован своими словами.**

### EDA (Exploratory Data Analysis)

**Своими словами:** EDA — это знакомство с данными до обучения модели. На этом этапе проверяют размер таблицы, типы столбцов, пропуски, выбросы, баланс целевого класса и связи между признаками. Графики помогают увидеть закономерности, которые трудно заметить в числовой таблице. Результат EDA — не набор красивых диаграмм, а обоснованный план очистки и построения признаков. EDA также помогает вовремя обнаружить ошибки и утечку целевой информации.

### Пропуски (NaN)

**Своими словами:** пропуски возникают, когда сведения не были собраны, потерялись или неприменимы к объекту. Большинство классических алгоритмов sklearn не принимает NaN напрямую. Небольшое число пропусков можно заполнить статистикой, а почти пустой столбец иногда разумнее удалить. Медиана устойчивее среднего к выбросам, а заполнение внутри осмысленных групп лучше сохраняет структуру данных. Стратегию нужно выбирать по смыслу признака, а не только по проценту NaN.

### Кодирование категорий

**Своими словами:** модели работают с числами, поэтому текстовые категории кодируют. Label Encoding назначает категории числовые метки и удобен для бинарного признака, но для нескольких номинальных категорий может создать ложный порядок. One-Hot Encoding создаёт отдельный индикатор для каждой категории и не навязывает отношения «больше — меньше». Цена one-hot — увеличение числа столбцов.

### Масштабирование

**Своими словами:** признаки могут измеряться в разных единицах и иметь сильно разные диапазоны. StandardScaler вычитает среднее и делит на стандартное отклонение, поэтому данные получают среднее около нуля и стандартное отклонение около единицы. MinMaxScaler переносит значения в заданный диапазон, обычно от 0 до 1, но сильнее зависит от выбросов. Логистической регрессии масштабирование полезно, а дереву решений оно не требуется, поскольку дерево сравнивает значения с порогами.

### Feature Engineering

**Своими словами:** feature engineering превращает исходные столбцы в признаки, которые лучше выражают смысл задачи. Например, суммы `SibSp` и `Parch` недостаточно, пока не учтён сам пассажир; так появляется размер семьи. Новый признак может сделать закономерность проще для модели и повысить интерпретируемость. Создавать его нужно только из информации, доступной в момент будущего прогноза.

### Метрики классификации

**Своими словами:** accuracy показывает долю всех правильных ответов, но может вводить в заблуждение при дисбалансе классов. Precision отвечает, как часто прогноз положительного класса верен. Recall показывает, какую долю реальных положительных объектов модель нашла. F1 объединяет precision и recall гармоническим средним и полезна, когда важны обе ошибки. ROC-AUC оценивает способность модели ставить положительные объекты выше отрицательных по вероятности и не привязан к одному порогу. Поэтому модели сравниваются по нескольким метрикам.

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

$$Precision = \frac{TP}{TP + FP} \qquad Recall = \frac{TP}{TP + FN}$$

$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

$$TPR = \frac{TP}{TP+FN} \qquad FPR = \frac{FP}{FP+TN}$$

| Метрика | Главный вопрос | Особенность |
|---|---|---|
| Accuracy | Сколько ответов верно вообще? | Зависит от баланса классов |
| Precision | Можно ли доверять прогнозу «выжил»? | Штрафует ложноположительные ответы |
| Recall | Скольких выживших нашли? | Штрафует пропущенные положительные ответы |
| F1 | Есть ли баланс precision и recall? | Гармоническое среднее |
| ROC-AUC | Хорошо ли ранжируются вероятности? | Не зависит от одного порога |

## Описание данных

**Источник:** [Kaggle — Titanic: Machine Learning from Disaster](https://www.kaggle.com/c/titanic)

Тренировочная часть содержит 891 пассажира и 11 входных столбцов плюс целевую переменную `Survived`; тестовая часть содержит 418 пассажиров без целевой переменной. Среди исходных входов 5 числовых признаков (`Age`, `SibSp`, `Parch`, `Fare`, `PassengerId`), один порядковый числовой (`Pclass`) и 5 категориальных/текстовых (`Name`, `Sex`, `Ticket`, `Cabin`, `Embarked`). Идентификатор, имя и номер билета не включаются в базовую модель.

| Столбец | Тип | Описание | Пропуски в train |
|---|---|---|---:|
| PassengerId | int64 | Идентификатор пассажира | 0 |
| Survived | int64 (0/1) | Целевая переменная | 0 |
| Pclass | int64 (1–3) | Класс билета | 0 |
| Name | object | Имя пассажира | 0 |
| Sex | object | Пол | 0 |
| Age | float64 | Возраст | 177 |
| SibSp | int64 | Супруги, братья и сёстры на борту | 0 |
| Parch | int64 | Родители и дети на борту | 0 |
| Ticket | object | Номер билета | 0 |
| Fare | float64 | Стоимость билета | 0 |
| Cabin | object | Каюта | 687 |
| Embarked | object | Порт посадки: C/Q/S | 2 |

Распределение классов: 549 (61,6%) не выжили и 342 (38,4%) выжили. Это умеренный дисбаланс.

In [1]:
# Базовые библиотеки и воспроизводимые настройки.
from pathlib import Path
from shutil import copy2
import json
import platform
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42

# В репозитории данные должны лежать в data/. Если исходные CSV пока рядом
# с ноутбуком, один раз копируем их в требуемую структуру.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
EXAMPLES_DIR = Path("examples")
EXAMPLES_DIR.mkdir(exist_ok=True)
for filename in ("train.csv", "test.csv"):
    source = Path(filename)
    destination = DATA_DIR / filename
    if not destination.exists() and source.exists():
        copy2(source, destination)

# Загружаем обе части и сразу проверяем формы.
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")
print(train_raw.shape, test_raw.shape)
display(train_raw.head())

# Создаём требуемое описание данных в папке data/.
class_counts = train_raw["Survived"].value_counts().sort_index()
info_text = f'''# Titanic dataset

Источник: https://www.kaggle.com/c/titanic

- train: {train_raw.shape[0]} строк, {train_raw.shape[1]} столбцов
- test: {test_raw.shape[0]} строк, {test_raw.shape[1]} столбцов
- класс 0: {class_counts[0]} ({class_counts[0] / len(train_raw):.1%})
- класс 1: {class_counts[1]} ({class_counts[1] / len(train_raw):.1%})

Пропуски train:
{train_raw.isna().sum().to_string()}
'''
(DATA_DIR / "titanic_info.md").write_text(info_text, encoding="utf-8")
print("Создан файл:", DATA_DIR / "titanic_info.md")

(891, 12) (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Создан файл: data\titanic_info.md


## Подготовка данных и EDA

Сначала выполняется первичный осмотр, затем статистический анализ и визуализация. Графики строятся по исходным данным, чтобы заполнение пропусков не скрывало реальную картину.

In [2]:
# Первичный осмотр структуры, типов и описательных статистик.
print("Размер train:", train_raw.shape)
display(train_raw.head())
train_raw.info()
display(train_raw.describe(include="all").T)

# Таблица пропусков в абсолютных значениях и процентах.
missing_table = pd.DataFrame({
    "missing": train_raw.isna().sum(),
    "percent": train_raw.isna().mean().mul(100).round(2),
}).query("missing > 0").sort_values("percent", ascending=False)
display(missing_table)

# Статистики формы распределения и частоты категорий.
numeric_cols = train_raw.select_dtypes(include=np.number).columns
shape_stats = pd.DataFrame({
    "skew": train_raw[numeric_cols].skew(),
    "kurtosis": train_raw[numeric_cols].kurtosis(),
})
display(shape_stats)
for column in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(f"\n{column}:\n{train_raw[column].value_counts(dropna=False)}")

Размер train: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PassengerId,891.0,NaN,NaN,NaN,446.0,257.353842,1.0,223.5,446.0,668.5,891.0
Survived,891.0,NaN,NaN,NaN,0.383838,0.486592,0.0,0.0,0.0,1.0,1.0
Pclass,891.0,NaN,NaN,NaN,2.308642,0.836071,1.0,2.0,3.0,3.0,3.0
Name,891,891,"Braund, Mr. Owen Harris",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,891,2,male,577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,714.0,NaN,NaN,NaN,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0
SibSp,891.0,NaN,NaN,NaN,0.523008,1.102743,0.0,0.0,0.0,1.0,8.0
Parch,891.0,NaN,NaN,NaN,0.381594,0.806057,0.0,0.0,0.0,0.0,6.0
Ticket,891,681,347082,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,891.0,NaN,NaN,NaN,32.204208,49.693429,0.0,7.9104,14.4542,31.0,512.3292


,missing,percent
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


,skew,kurtosis
PassengerId,0.000000,-1.200000
Survived,0.478523,-1.775005
Pclass,-0.630548,-1.280015
Age,0.389108,0.178274
SibSp,3.695352,17.880420
Parch,2.749117,9.778125
Fare,4.787317,33.398141



Survived:
Survived
0    549
1    342
Name: count, dtype: int64

Pclass:
Pclass
3    491
1    216
2    184
Name: count, dtype: int64

Sex:
Sex
male      577
female    314
Name: count, dtype: int64

Embarked:
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64


In [3]:
# Семь визуальных срезов: классы, возраст, boxplot, пропуски,
# корреляции и выживаемость по трём категориальным признакам.
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
sns.countplot(data=train_raw, x="Survived", ax=axes[0, 0])
axes[0, 0].set(title="Распределение целевого класса", xlabel="Выжил (0/1)", ylabel="Число пассажиров")

sns.histplot(data=train_raw, x="Age", hue="Survived", kde=True, bins=30, ax=axes[0, 1])
axes[0, 1].set(title="Возраст и выживание", xlabel="Возраст, лет", ylabel="Частота")

sns.boxplot(data=train_raw, x="Pclass", y="Age", hue="Survived", ax=axes[0, 2])
axes[0, 2].set(title="Возраст по классу и исходу", xlabel="Класс билета", ylabel="Возраст, лет")

missing_plot = train_raw.isna().mean().mul(100).sort_values(ascending=False)
sns.barplot(x=missing_plot.values, y=missing_plot.index, ax=axes[1, 0], color="steelblue")
axes[1, 0].set(title="Доля пропусков", xlabel="Пропуски, %", ylabel="Столбец")

corr = train_raw.select_dtypes(include=np.number).corr()
sns.heatmap(corr, cmap="coolwarm", center=0, ax=axes[1, 1])
axes[1, 1].set_title("Корреляции числовых признаков")

survival_by_sex = train_raw.groupby("Sex", as_index=False)["Survived"].mean()
sns.barplot(data=survival_by_sex, x="Sex", y="Survived", ax=axes[1, 2])
axes[1, 2].set(title="Выживаемость по полу", xlabel="Пол", ylabel="Доля выживших")
plt.tight_layout()
fig.savefig(EXAMPLES_DIR / "eda_plots.png", dpi=150, bbox_inches="tight")
plt.show()

# Ещё два обязательных сравнения: класс билета и порт посадки.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for axis, column, title in zip(axes, ["Pclass", "Embarked"], ["классу билета", "порту посадки"]):
    rates = train_raw.groupby(column, as_index=False)["Survived"].mean()
    sns.barplot(data=rates, x=column, y="Survived", ax=axis)
    axis.set(title=f"Выживаемость по {title}", xlabel=column, ylabel="Доля выживших")
plt.tight_layout()
fig.savefig(EXAMPLES_DIR / "survival_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

### Наблюдения по EDA

- Не выживших заметно больше, поэтому accuracy рассматривается вместе с F1 и ROC-AUC.
- Выживаемость женщин существенно выше выживаемости мужчин.
- Пассажиры первого класса выживали чаще пассажиров третьего класса.
- Распределение `Fare` имеет сильную правую асимметрию и дорогие выбросы.
- `Cabin` почти пуст, а в `Age` отсутствует примерно пятая часть значений.

### Обоснование обработки пропусков

**Своими словами:** `Age` заполняется медианой внутри сочетания `Pclass × Sex`: медиана устойчива к выбросам, а группы учитывают различия состава пассажиров. Два пропуска `Embarked` заполняются модой, потому что это номинальный признак и доля пропусков мала. `Cabin` удаляется из базовой модели из-за примерно 77% неизвестных значений; достоверно восстановить номер каюты нельзя. В тестовой части единственный пропуск `Fare` заполняется медианой тренировочной выборки, чтобы не использовать статистику будущих данных.

In [4]:
from sklearn.preprocessing import LabelEncoder

# Работаем с копией, сохраняя исходный DataFrame для проверки и графиков.
train_df = train_raw.copy()
test_df = test_raw.copy()

# Медианы возраста вычисляем только на train и применяем к обоим наборам.
age_medians = train_df.groupby(["Pclass", "Sex"])["Age"].median()
def fill_age(frame):
    frame = frame.copy()
    missing = frame["Age"].isna()
    frame.loc[missing, "Age"] = frame.loc[missing].apply(
        lambda row: age_medians.loc[(row["Pclass"], row["Sex"])], axis=1
    )
    return frame

train_df = fill_age(train_df)
test_df = fill_age(test_df)

# Остальные заполнения также используют статистики train.
embarked_mode = train_df["Embarked"].mode()[0]
fare_median = train_df["Fare"].median()
train_df["Embarked"] = train_df["Embarked"].fillna(embarked_mode)
test_df["Embarked"] = test_df["Embarked"].fillna(embarked_mode)
test_df["Fare"] = test_df["Fare"].fillna(fare_median)
train_df = train_df.drop(columns=["Cabin"])
test_df = test_df.drop(columns=["Cabin"])

# Создаём размер семьи, признак одиночного путешествия и возрастную группу.
for frame in (train_df, test_df):
    frame["Family_Size"] = frame["SibSp"] + frame["Parch"] + 1
    frame["Is_Alone"] = (frame["Family_Size"] == 1).astype(int)
    frame["Age_Group"] = pd.cut(
        frame["Age"], bins=[0, 12, 18, 35, 60, np.inf],
        labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"], include_lowest=True
    )

# Бинарный Sex кодируем LabelEncoder, Embarked — one-hot.
le_sex = LabelEncoder()
train_df["Sex_Encoded"] = le_sex.fit_transform(train_df["Sex"])
test_df["Sex_Encoded"] = le_sex.transform(test_df["Sex"])
train_df = pd.get_dummies(train_df, columns=["Embarked"], prefix="Embarked", drop_first=True, dtype=int)
test_df = pd.get_dummies(test_df, columns=["Embarked"], prefix="Embarked", drop_first=True, dtype=int)

print("Кодировка Sex:", dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_))))
print("Осталось пропусков в модельных признаках:", train_df.isna().sum().sum())
display(train_df[["Family_Size", "Is_Alone", "Age_Group", "Sex_Encoded", "Embarked_Q", "Embarked_S"]].head())

Кодировка Sex: {'female': np.int64(0), 'male': np.int64(1)}
Осталось пропусков в модельных признаках: 0


,Family_Size,Is_Alone,Age_Group,Sex_Encoded,Embarked_Q,Embarked_S
0,2,0,YoungAdult,1,0,1
1,2,0,Adult,0,0,0
2,1,1,YoungAdult,0,0,1
3,2,0,YoungAdult,0,0,1
4,1,1,YoungAdult,1,0,1


## Построение и обучение моделей

Logistic Regression служит интерпретируемой линейной базой, Decision Tree улавливает нелинейные правила, а Random Forest добавлен как бонусный ансамбль. Разделение стратифицировано, чтобы сохранить долю выживших. Масштабирование обучается только на тренировочной части и применяется к логистической регрессии; деревьям оно не нужно.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Признаки доступны и для train, и для нового пассажира.
feature_cols = [
    "Pclass", "Sex_Encoded", "Age", "SibSp", "Parch", "Fare",
    "Family_Size", "Is_Alone", "Embarked_Q", "Embarked_S",
]
X = train_df[feature_cols].astype(float)
y = train_df["Survived"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Масштабируем без утечки статистик validation в train.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Инициализируем модели с фиксированным random_state.
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
dt_model = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE)
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=3,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
)

# Замеряем время обучения каждой модели.
training_times = {}
for name, model, features in [
    ("Logistic Regression", lr_model, X_train_scaled),
    ("Decision Tree", dt_model, X_train),
    ("Random Forest", rf_model, X_train),
]:
    started = time.perf_counter()
    model.fit(features, y_train)
    training_times[name] = time.perf_counter() - started
    print(f"{name}: {training_times[name]:.4f} с")

Logistic Regression: 0.0108 с
Decision Tree: 0.0060 с


Random Forest: 0.7292 с


## Оценка качества

Для каждой модели рассчитываются Accuracy, Precision, Recall, F1 и ROC-AUC. Матрицы ошибок показывают типы ошибок при пороге 0,5, а ROC-кривые позволяют сравнить ранжирование вероятностей на всех порогах.

In [6]:
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve,
)

# Получаем классы и вероятности на одной и той же validation-выборке.
model_specs = {
    "Logistic Regression": (lr_model, X_valid_scaled),
    "Decision Tree": (dt_model, X_valid),
    "Random Forest": (rf_model, X_valid),
}
predictions = {}
metrics_rows = []
for name, (model, features) in model_specs.items():
    y_pred = model.predict(features)
    y_prob = model.predict_proba(features)[:, 1]
    predictions[name] = {"class": y_pred, "probability": y_prob}
    metrics_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_valid, y_pred),
        "Precision": precision_score(y_valid, y_pred),
        "Recall": recall_score(y_valid, y_pred),
        "F1": f1_score(y_valid, y_pred),
        "ROC-AUC": roc_auc_score(y_valid, y_prob),
        "Train time, s": training_times[name],
    })

results_table = pd.DataFrame(metrics_rows).set_index("Model").sort_values("ROC-AUC", ascending=False)
display(results_table.style.format("{:.4f}"))

# Матрицы ошибок для всех сравниваемых моделей.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, (name, values) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_valid, values["class"])).plot(
        ax=axis, colorbar=False, cmap="Blues"
    )
    axis.set_title(name)
plt.tight_layout()
fig.savefig(EXAMPLES_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# ROC-кривые и диагональ случайного классификатора.
roc_fig = plt.figure(figsize=(8, 6))
for name, values in predictions.items():
    fpr, tpr, _ = roc_curve(y_valid, values["probability"])
    auc_value = roc_auc_score(y_valid, values["probability"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_value:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Случайная модель")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривые на validation-выборке")
plt.legend()
roc_fig.savefig(EXAMPLES_DIR / "roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

,Accuracy,Precision,Recall,F1,ROC-AUC,"Train time, s"
Model,,,,,,
Logistic Regression,0.8045,0.7833,0.6812,0.7287,0.8486,0.0108
Random Forest,0.7933,0.7222,0.7536,0.7376,0.8483,0.7292
Decision Tree,0.7654,0.7077,0.6667,0.6866,0.8031,0.0060


## Интерпретация результатов

Коэффициенты логистической регрессии интерпретируются по знаку: положительный коэффициент увеличивает log-odds выживания, отрицательный — уменьшает. Поскольку признаки стандартизованы, абсолютные величины коэффициентов можно приблизительно сравнивать. Для деревьев используется уменьшение impurity, суммированное по разбиениям с данным признаком.

In [7]:
# Коэффициенты линейной модели и важности двух деревьев.
lr_importance = pd.Series(lr_model.coef_[0], index=feature_cols).sort_values()
dt_importance = pd.Series(dt_model.feature_importances_, index=feature_cols).sort_values()
rf_importance = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
lr_importance.plot.barh(ax=axes[0], title="Коэффициенты Logistic Regression")
dt_importance.plot.barh(ax=axes[1], title="Feature importance Decision Tree")
rf_importance.plot.barh(ax=axes[2], title="Feature importance Random Forest")
for axis in axes:
    axis.set_xlabel("Вклад / важность")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "LR coefficient": lr_importance,
    "DT importance": dt_importance,
    "RF importance": rf_importance,
}).sort_values("RF importance", ascending=False))

,LR coefficient,DT importance,RF importance
Sex_Encoded,-1.225895,0.554905,0.394254
Fare,0.077219,0.102348,0.189004
Age,-0.557201,0.102337,0.135158
Pclass,-0.971323,0.193020,0.118223
Family_Size,-0.223771,0.045690,0.061108
SibSp,-0.294769,0.001700,0.028327
Embarked_S,-0.129193,0.000000,0.027228
Parch,-0.051203,0.000000,0.019817
Is_Alone,-0.283692,0.000000,0.016169
Embarked_Q,0.088102,0.000000,0.010712


### Бизнес-инсайты

**Своими словами:** пол и класс билета связаны с доступом к спасательным шлюпкам и поэтому дают сильный сигнал, но это историческая связь, а не причинное доказательство. Более высокая стоимость билета частично отражает класс и расположение пассажира. Размер семьи полезен нелинейно: одиночество и очень большая семья могли осложнять эвакуацию. Возраст особенно важен для детей, поэтому простой линейный коэффициент может показывать эффект слабее дерева. Эти выводы относятся к конкретной катастрофе и не должны переноситься на современные решения о безопасности без дополнительного исследования.

## Функция предсказания

Функция принимает понятные человеку поля, повторяет те же преобразования, что использовались при обучении, и возвращает класс и вероятность. По умолчанию используется модель с максимальным ROC-AUC на validation-выборке.

In [8]:
def prepare_passenger(passenger):
    '''Преобразовать словарь с сырыми полями в строку модельных признаков.'''
    family_size = passenger["SibSp"] + passenger["Parch"] + 1
    sex_encoded = int(le_sex.transform([passenger["Sex"]])[0])
    row = {
        "Pclass": passenger["Pclass"], "Sex_Encoded": sex_encoded,
        "Age": passenger["Age"], "SibSp": passenger["SibSp"],
        "Parch": passenger["Parch"], "Fare": passenger["Fare"],
        "Family_Size": family_size, "Is_Alone": int(family_size == 1),
        "Embarked_Q": int(passenger["Embarked"] == "Q"),
        "Embarked_S": int(passenger["Embarked"] == "S"),
    }
    return pd.DataFrame([row], columns=feature_cols).astype(float)

def predict_passenger(passenger, model_name=None):
    '''Вернуть прогноз (0/1), вероятность выживания и имя модели.'''
    selected_name = model_name or results_table["ROC-AUC"].idxmax()
    model_lookup = {
        "Logistic Regression": lr_model,
        "Decision Tree": dt_model,
        "Random Forest": rf_model,
    }
    row = prepare_passenger(passenger)
    model_input = scaler.transform(row) if selected_name == "Logistic Regression" else row
    probability = float(model_lookup[selected_name].predict_proba(model_input)[0, 1])
    return {"prediction": int(probability >= 0.5), "probability": probability, "model": selected_name}

# Пять пассажиров с разными сочетаниями пола, класса, возраста и семьи.
demo_passengers = [
    {"Pclass": 1, "Sex": "female", "Age": 29, "SibSp": 0, "Parch": 0, "Fare": 100, "Embarked": "C"},
    {"Pclass": 3, "Sex": "male", "Age": 35, "SibSp": 0, "Parch": 0, "Fare": 8, "Embarked": "S"},
    {"Pclass": 2, "Sex": "female", "Age": 8, "SibSp": 1, "Parch": 2, "Fare": 30, "Embarked": "S"},
    {"Pclass": 1, "Sex": "male", "Age": 54, "SibSp": 1, "Parch": 0, "Fare": 80, "Embarked": "C"},
    {"Pclass": 3, "Sex": "female", "Age": 22, "SibSp": 0, "Parch": 0, "Fare": 7.5, "Embarked": "Q"},
]
demo_results = []
for number, passenger in enumerate(demo_passengers, start=1):
    result = predict_passenger(passenger)
    demo_results.append(result)
    print(f"Пассажир {number}: класс={result['prediction']}, p={result['probability']:.3f}, модель={result['model']}")

Пассажир 1: класс=1, p=0.947, модель=Logistic Regression
Пассажир 2: класс=0, p=0.063, модель=Logistic Regression
Пассажир 3: класс=1, p=0.875, модель=Logistic Regression
Пассажир 4: класс=0, p=0.354, модель=Logistic Regression
Пассажир 5: класс=1, p=0.735, модель=Logistic Regression


## Сохранение модели

Сохраняются обе обязательные модели, бонусный ансамбль, scaler, кодировщик, список признаков, метрики и метаданные среды. Такой набор позволяет воспроизвести ровно те же преобразования при инференсе.

In [9]:
# Создаём стандартные папки артефактов модуля.
MODELS_DIR = Path("models")
EXAMPLES_DIR = Path("examples")
MODELS_DIR.mkdir(exist_ok=True)
EXAMPLES_DIR.mkdir(exist_ok=True)

# Бинарные объекты sklearn сохраняем joblib.
joblib.dump(lr_model, MODELS_DIR / "lr_model.pkl")
joblib.dump(dt_model, MODELS_DIR / "dt_model.pkl")
joblib.dump(rf_model, MODELS_DIR / "rf_model.pkl")
joblib.dump(scaler, MODELS_DIR / "scaler.pkl")
joblib.dump(le_sex, MODELS_DIR / "le_sex.pkl")

# JSON хранит переносимые сведения без Python-объектов.
(MODELS_DIR / "feature_cols.json").write_text(
    json.dumps(feature_cols, ensure_ascii=False, indent=2), encoding="utf-8"
)
metrics_json = {
    model: {metric: float(value) for metric, value in row.items()}
    for model, row in results_table.to_dict(orient="index").items()
}
(MODELS_DIR / "metrics.json").write_text(
    json.dumps(metrics_json, ensure_ascii=False, indent=2), encoding="utf-8"
)
metadata = {
    "created": "2026-09-22", "python": platform.python_version(),
    "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
    "random_state": RANDOM_STATE, "target": "Survived",
    "sex_classes": le_sex.classes_.tolist(),
}
(MODELS_DIR / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Сохранено:", sorted(path.name for path in MODELS_DIR.iterdir()))

Сохранено: ['dt_model.pkl', 'feature_cols.json', 'le_sex.pkl', 'lr_model.pkl', 'metadata.json', 'metrics.json', 'rf_model.pkl', 'scaler.pkl']


## Загрузка модели из репозитория

Для реальной загрузки укажите raw-адрес папки модуля в `GITHUB_RAW_BASE`, например `https://raw.githubusercontent.com/USER/ml-course-surname/main/module-1-titanic`. Пока URL не указан, ячейка делает эквивалентную локальную загрузку и проверяет совпадение прогнозов на трёх пассажирах. После публикации достаточно заменить одну строку — остальная логика не меняется.

In [10]:
from io import BytesIO
import requests

# Вставьте raw URL после создания публичного репозитория.
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Ogn1k/ml-course-fedorushkin/main/module-1-titanic"

if GITHUB_RAW_BASE:
    # Загружаем бинарные объекты и JSON по HTTPS с проверкой статуса.
    def load_remote_joblib(relative_path):
        response = requests.get(f"{GITHUB_RAW_BASE}/{relative_path}", timeout=30)
        response.raise_for_status()
        return joblib.load(BytesIO(response.content))
    loaded_models = {
        "Logistic Regression": load_remote_joblib("models/lr_model.pkl"),
        "Decision Tree": load_remote_joblib("models/dt_model.pkl"),
        "Random Forest": load_remote_joblib("models/rf_model.pkl"),
    }
    loaded_scaler = load_remote_joblib("models/scaler.pkl")
    loaded_features = requests.get(
        f"{GITHUB_RAW_BASE}/models/feature_cols.json", timeout=30
    ).json()
    source_label = "GitHub raw"
else:
    # Локальный round-trip проверяет сериализацию до публикации.
    loaded_models = {
        "Logistic Regression": joblib.load(MODELS_DIR / "lr_model.pkl"),
        "Decision Tree": joblib.load(MODELS_DIR / "dt_model.pkl"),
        "Random Forest": joblib.load(MODELS_DIR / "rf_model.pkl"),
    }
    loaded_scaler = joblib.load(MODELS_DIR / "scaler.pkl")
    loaded_features = json.loads((MODELS_DIR / "feature_cols.json").read_text(encoding="utf-8"))
    source_label = "локальные сохранённые файлы"

# Сравниваем вероятности загруженной и исходной модели на трёх примерах.
best_model_name = results_table["ROC-AUC"].idxmax()
for number, passenger in enumerate(demo_passengers[:3], start=1):
    row = prepare_passenger(passenger)[loaded_features]
    loaded_input = loaded_scaler.transform(row) if best_model_name == "Logistic Regression" else row
    loaded_probability = float(loaded_models[best_model_name].predict_proba(loaded_input)[0, 1])
    original_probability = demo_results[number - 1]["probability"]
    assert np.isclose(loaded_probability, original_probability), "Предсказания не совпали"
    print(f"Пример {number}: {loaded_probability:.6f} — совпадает")
print("Источник загрузки:", source_label)

Пример 1: 0.946732 — совпадает
Пример 2: 0.062669 — совпадает
Пример 3: 0.875078 — совпадает
Источник загрузки: локальные сохранённые файлы


## Выводы

Следующая ячейка формирует вывод из фактически рассчитанных метрик, поэтому числа не расходятся с результатами запуска.

In [11]:
from IPython.display import Markdown, display

# Подставляем фактические значения лучшей модели в итоговый текст.
best_name = results_table["ROC-AUC"].idxmax()
best = results_table.loc[best_name]
display(Markdown(f'''
### Что получилось

- Достигнут ROC-AUC **{best['ROC-AUC']:.3f}** на отложенной выборке.
- Лучшая по ROC-AUC модель: **{best_name}**; Accuracy = **{best['Accuracy']:.3f}**, F1 = **{best['F1']:.3f}**.
- Время её обучения: **{best['Train time, s']:.4f} с** на текущем компьютере.
- Главные наблюдения EDA: более высокая выживаемость женщин и пассажиров высоких классов; `Cabin` имеет критически много пропусков.

### Трудности

- Около 20% значений `Age` пришлось восстанавливать групповыми медианами.
- `Cabin` содержит около 77% пропусков и исключён из базовой модели.
- Классы умеренно несбалансированы: 61,6% против 38,4%.
- Глубина дерева ограничена, чтобы уменьшить переобучение.
- Удалённая проверка GitHub станет доступна только после публикации репозитория и заполнения `GITHUB_RAW_BASE`.

### Возможные улучшения

1. Извлечь обращение (`Title`: Mr, Mrs, Miss и т. п.) из `Name`.
2. Добавить взаимодействие `Pclass × Sex`.
3. Проверить SMOTE только внутри обучающих фолдов.
4. Настроить Random Forest и градиентный бустинг.
5. Подобрать гиперпараметры через `GridSearchCV` со стратифицированной кросс-валидацией.
6. Применить `np.log1p(Fare)` и сравнить метрики.
7. Настроить порог классификации под цену ошибок, а не всегда использовать 0,5.
'''))


### Что получилось

- Достигнут ROC-AUC **0.849** на отложенной выборке.
- Лучшая по ROC-AUC модель: **Logistic Regression**; Accuracy = **0.804**, F1 = **0.729**.
- Время её обучения: **0.0108 с** на текущем компьютере.
- Главные наблюдения EDA: более высокая выживаемость женщин и пассажиров высоких классов; `Cabin` имеет критически много пропусков.

### Трудности

- Около 20% значений `Age` пришлось восстанавливать групповыми медианами.
- `Cabin` содержит около 77% пропусков и исключён из базовой модели.
- Классы умеренно несбалансированы: 61,6% против 38,4%.
- Глубина дерева ограничена, чтобы уменьшить переобучение.
- Удалённая проверка GitHub станет доступна только после публикации репозитория и заполнения `GITHUB_RAW_BASE`.

### Возможные улучшения

1. Извлечь обращение (`Title`: Mr, Mrs, Miss и т. п.) из `Name`.
2. Добавить взаимодействие `Pclass × Sex`.
3. Проверить SMOTE только внутри обучающих фолдов.
4. Настроить Random Forest и градиентный бустинг.
5. Подобрать гиперпараметры через `GridSearchCV` со стратифицированной кросс-валидацией.
6. Применить `np.log1p(Fare)` и сравнить метрики.
7. Настроить порог классификации под цену ошибок, а не всегда использовать 0,5.


## Источники

1. Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.
2. McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media.
3. Kaggle. [Titanic — Machine Learning from Disaster](https://www.kaggle.com/c/titanic).
4. Scikit-learn. [User Guide](https://scikit-learn.org/stable/user_guide.html).
5. pandas. [User Guide](https://pandas.pydata.org/docs/user_guide/index.html).
6. VanderPlas, J. (2016). *Python Data Science Handbook*. O'Reilly Media.
7. Материалы курса: `Теория/Модуль 1. Введение в Data Science и EDA.ipynb`.

## Приложение

Все исполняемые ячейки выше образуют полный воспроизводимый код. Ячейка ниже автоматически экспортирует их в Jupytext-подобный файл с разделителями `# %%`, чтобы код был собран в одном месте.

Зависимости (`requirements.txt`):

```text
numpy>=1.24
pandas>=2.0
matplotlib>=3.7
seaborn>=0.12
scikit-learn>=1.3
joblib>=1.3
requests>=2.31
nbformat>=5.9
```

In [12]:
# Записываем зависимости, необходимые для воспроизведения проекта.
requirements = '''numpy>=1.24
pandas>=2.0
matplotlib>=3.7
seaborn>=0.12
scikit-learn>=1.3
joblib>=1.3
requests>=2.31
nbformat>=5.9
'''
Path("requirements.txt").write_text(requirements, encoding="utf-8")

# Собираем полный код из текущего ноутбука с разделителями # %%.
notebook_candidates = [
    Path("Модуль 1. Практическое задание 1.ipynb"),
    Path("notebook.ipynb"),
]
notebook_path = next(path for path in notebook_candidates if path.exists())
current_notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
script_parts = []
for cell in current_notebook["cells"]:
    if cell["cell_type"] == "code":
        script_parts.append("# %%\n" + "".join(cell.get("source", [])))
full_script = "\n\n".join(script_parts)
Path("appendix_full_code.py").write_text(full_script, encoding="utf-8")
print("Созданы requirements.txt и appendix_full_code.py")
print("Число экспортированных кодовых ячеек:", len(script_parts))

Созданы requirements.txt и appendix_full_code.py
Число экспортированных кодовых ячеек: 12
